In [ ]:
import time
PHASE_START=time.monotonic()
from pathlib import Path
import sys,os,json,subprocess,hashlib,zipfile
from google.colab import files
base=Path('/content/ns64_clean')
blocked=['/content/drive/MyDrive','/content/ns64','/content/imc_ns64_colab_migration.zip','/content/imc_ets_images.zip','/content/private.json']
assert not any(Path(p).exists() for p in blocked),'Use a freshly deleted and reconnected runtime without Drive'
subprocess.run(['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader'],check=True)
bundle=Path('/content/imc_ns64_clean_runtime.zip')
if not bundle.exists(): files.upload()
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='39e9bb97ede83c777a06f385fbe4acb27fc4665ba1556b504dc64000dcafb925','Wrong runtime ZIP'
base.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:z.extractall(base)
bootstrap=base/'bootstrap'
subprocess.run([sys.executable,'-m','pip','install','--target',str(bootstrap),'uv==0.13.8'],check=True)
setup_env=os.environ.copy()
setup_env.update(PYTHONPATH=str(bootstrap),UV_PYTHON_INSTALL_DIR=str(base/'python'),UV_CACHE_DIR=str(base/'uv_cache'))
uv=[sys.executable,'-m','uv']
subprocess.run(uv+['python','install','3.11.11'],env=setup_env,check=True)
python=base/'venv/bin/python'
if not python.exists():subprocess.run(uv+['venv','--python','3.11.11','--seed',str(base/'venv')],env=setup_env,check=True)
subprocess.run([str(python),'-m','pip','install','setuptools==75.8.0','wheel==0.45.1'],check=True)

subprocess.run([str(python),'-m','pip','install','--no-deps','--extra-index-url','https://download.pytorch.org/whl/cu121','-r',str(base/'requirements.txt')],check=True)


In [ ]:
from google.colab import files
work=Path('/content/phase4f')
work.mkdir()
print('Upload phase4f_colab_worker.zip, imc2023_haiper_opaque.zip and the hash-named .db together.')
files.upload()
kit=Path('/content/phase4f_colab_worker.zip')
assert hashlib.sha256(kit.read_bytes()).hexdigest()=='75dff77c2107d3637a1792658f26fbd9732ae573a08542a1ce7673c49ed8a199'
with zipfile.ZipFile(kit) as z: z.extractall(work)
plan=json.loads((work/'plan.json').read_bytes())
parent=Path('/content')/(plan['parent']+'.db')
assert parent.is_file()
actual=subprocess.check_output([str(python),'-m','pip','freeze'],text=True)
assert sorted(actual.splitlines())==sorted((work/'expected_freeze.txt').read_text().splitlines()), 'Environment differs from 4C'
images=work/'images'
images.mkdir()
with zipfile.ZipFile('/content/imc2023_haiper_opaque.zip') as z:
    assert sorted(z.namelist())==sorted(plan['image_hashes'])
    z.extractall(images)


In [ ]:
def execute(branch):
    remain=int(5400-(time.monotonic()-PHASE_START))-90
    assert remain>0, 'Combined time budget exhausted'
    log=work/(branch+'.log')
    assert not log.exists(), 'This branch was already attempted'
    status='FAILED'
    try:
        with log.open('w') as f:
            result=subprocess.run([str(python),'-m','scripts.phase4f_run_mapper_comparison',branch,str(parent),str(images)],cwd=work,stdout=f,stderr=subprocess.STDOUT,timeout=remain)
        status='PASS' if result.returncode==0 else 'FAILED'
    except subprocess.TimeoutExpired:
        status='TIMEOUT'
    finally:
        archive=Path('/content')/('phase4f_'+branch+'_return.zip')
        with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(log,log.name)
            z.writestr('status.json',json.dumps(dict(status=status,elapsed=time.monotonic()-PHASE_START)))
            for path in (work/branch).glob('*'):
                if path.suffix in ('.csv','.json'): z.write(path,path.name)
        files.download(str(archive))
        print(status,archive)
execute('pass_through')


In [ ]:
# Only after the independently evaluated pass-through return is approved.
print('Upload the administrator-produced parity_gate.json')
files.upload()
import shutil
shutil.copyfile('/content/parity_gate.json',work/'parity_gate.json')
execute('active')
